# Domain 3: Claude Code Configuration & Workflows
## Task Statement 3.6: CI/CD Pipeline Automation & Headless Execution
**Certification Weight**: 18% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand how to integrate Claude Code into automated CI/CD pipelines using non-interactive flags, structured JSON output formats, schema enforcement, and ephemeral sessions.

---

## 🏗️ Architecture: Non-Interactive CI Runner

```mermaid
flowchart TD
    Trigger["CI Trigger<br>(GitHub / GitLab PR)"] --> Runner["Ephemeral CI Runner<br>Environment"]
    
    Runner --> Command["claude -p 'Review PR changes'<br>--output-format json<br>--json-schema review.json<br>--max-tokens 4000"]
    
    Command --> Exec["Headless Claude Code<br>Execution"]
    Exec --> JSONOutput["Structured JSON Artifact"]
    
    JSONOutput --> Parser["CI Pipeline Gatekeeper"]
    Parser --> Eval{"Security Violations?"}
    
    Eval -->|"Yes: High/Critical"| PRBlock["Post PR Review Comments<br>Exit Code 1 (Fail Pipeline)"]
    Eval -->|"No: Approved"| PRPass["Post Approval Comment<br>Exit Code 0 (Pass Pipeline)"]
```

---

## 📋 Essential CLI Flags for CI/CD

| Flag | Exam Significance |
|---|---|
| `-p` / `--print` | **Mandatory** for CI. Runs non-interactively without prompting for TTY input; prints final output and exits. |
| `--output-format json` | Formats output as machine-readable JSON rather than terminal markdown or ANSI escape sequences. |
| `--json-schema <path>` | Enforces an explicit JSON schema on the response so automated CI scripts can reliably parse fields. |
| `--max-tokens <int>` | Caps total token generation to prevent runaway cloud runner costs on long diffs. |

### Critical Exam Rules:
1. **Never use interactive mode in CI**: Without `-p` or `--print`, the CI runner will hang indefinitely waiting for interactive stdin.
2. **Stateless Ephemeral Sessions**: CI jobs should start fresh and avoid sharing state across builds.
3. **Deterministic Exit Codes**: CI scripts evaluate exit status (0 = success, 1 = failure) to block or approve merge requests.

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Setup
import json
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 3.6 CI/CD Pipeline Automation.")

### 🤖 Step 2: Headless CI Runner Implementation
We implement the headless PR auditor that evaluates git patches against security rules and emits exit codes.

In [ ]:
# Step 2: Headless CI Auditor
@dataclass
class CIReviewFinding:
    rule_violated: str
    severity: str  # 'CRITICAL' | 'WARNING' | 'INFO'
    file_path: str
    line_snippet: str
    message: str


@dataclass
class CIReviewResult:
    status: str  # 'APPROVED' | 'CHANGES_REQUESTED'
    exit_code: int
    findings: List[CIReviewFinding]


class HeadlessCIRunner:
    def __init__(self, non_interactive: bool = True) -> None:
        self.non_interactive = non_interactive

    def run_pr_audit(self, git_diff_patch: str) -> CIReviewResult:
        if not self.non_interactive:
            raise RuntimeError("CI Runner must be executed with non-interactive=True (-p flag)!")

        findings = []
        for line in git_diff_patch.splitlines():
            clean = line.strip()
            if clean.startswith("+") and not clean.startswith("+++"):
                # Check for dynamic code execution
                if "eval(" in clean or "exec(" in clean:
                    findings.append(CIReviewFinding(
                        rule_violated="SEC001_NO_DYNAMIC_EXEC",
                        severity="CRITICAL",
                        file_path="src/executor.py",
                        line_snippet=clean,
                        message="Dynamic execution functions (eval/exec) are strictly prohibited.",
                    ))
                # Check for hardcoded credentials
                if "sk-ant-" in clean and not any(m in clean for m in ["mock", "test", "dummy"]):
                    findings.append(CIReviewFinding(
                        rule_violated="SEC002_NO_HARDCODED_SECRETS",
                        severity="CRITICAL",
                        file_path="src/config.py",
                        line_snippet=clean,
                        message="Hardcoded Anthropic API token detected in pull request patch.",
                    ))

        has_blocking = any(f.severity == "CRITICAL" for f in findings)
        return CIReviewResult(
            status="CHANGES_REQUESTED" if has_blocking else "APPROVED",
            exit_code=1 if has_blocking else 0,
            findings=findings,
        )

print(" HeadlessCIRunner initialized.")

### 🧪 Step 3: Interactive Simulation
1. Audit PR patch containing dangerous `eval()`.
2. Audit clean PR patch.
3. Emit structured JSON report.

In [ ]:
# Step 3: Simulation
runner = HeadlessCIRunner(non_interactive=True)

# 1. Unsafe patch audit
unsafe_patch = """
diff --git a/src/executor.py b/src/executor.py
+    result = eval(user_supplied_str)
"""
res_unsafe = runner.run_pr_audit(unsafe_patch)
print("Unsafe PR Status:", res_unsafe.status)
print("Unsafe PR Exit Code:", res_unsafe.exit_code)
print("Findings:", [asdict(f) for f in res_unsafe.findings])

# 2. Clean patch audit
clean_patch = """
diff --git a/src/math.py b/src/math.py
+    return int(a) + int(b)
"""
res_clean = runner.run_pr_audit(clean_patch)
print("\nClean PR Status:", res_clean.status)
print("Clean PR Exit Code:", res_clean.exit_code)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Unsafe patch triggers `CHANGES_REQUESTED` and exit code 1.
2. Clean patch triggers `APPROVED` and exit code 0.
3. Non-interactive flag enforcement prevents terminal hanging.

In [ ]:
# Step 4: Verification Checks
assert res_unsafe.exit_code == 1
assert res_unsafe.status == "CHANGES_REQUESTED"
assert res_unsafe.findings[0].rule_violated == "SEC001_NO_DYNAMIC_EXEC"

assert res_clean.exit_code == 0
assert res_clean.status == "APPROVED"
assert len(res_clean.findings) == 0

print(" All verification assertions passed! CI/CD pipeline automation verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: CI runners must invoke Claude Code with `-p` / `--non-interactive` to prevent hanging on interactive prompts.
- **Rule 2**: Return exit code 1 to fail the CI build when critical security violations are discovered.
- **Rule 3**: Emit findings as structured JSON for automated GitHub PR comment posting.